In [1]:
from database.manager import DatabaseManager
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter

db = DatabaseManager()
sznlty = Seasonality(db)
plotter = SeasonalityPlotter()

In [2]:
result = sznlty.outright_seasonality("CO", "Z", start_year=2014, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CO Z — outright seasonality")

In [10]:
result = sznlty.expression_seasonality("CO", "Z26- F27", start_year=2014, end_year=2026, window_days=500)
plotter.plot_seasonality(result, title="CO Z26 — seasonality")

In [4]:
from analysis.feature_creation import FeatureCreator
import pandas as pd

fc = FeatureCreator()
anomaly_results = fc.get_anomaly_years(result['combined'])

print("--- Seasonal Anomaly Report (30d Rolling Volatility Based) ---")
for bracket, years in anomaly_results.items():
    print(f"{bracket.upper():<15}: {years}")

# Validation: View the Average Volatility for the 10y bracket
historical_pool = sorted(result['combined'].columns)[:-1]
last_10y = historical_pool[-10:]

if len(last_10y) >= 6:
    # Calculate daily 30d SD for each year
    vols = result['combined'][last_10y].rolling(window=30, min_periods=1).std()
    # Average volatility score per year
    vol_scores = vols.mean().sort_values(ascending=False).to_frame(name='Avg 30d Volatility')
    
    print("\n--- Volatility Scores (Highest = Most Anomalous) ---")
    display(vol_scores)

--- Seasonal Anomaly Report (30d Rolling Volatility Based) ---
BRACKET_15Y    : [2020, 2022, 2023]
BRACKET_10Y    : [2022, 2023]
BRACKET_5Y     : [2022]

--- Volatility Scores (Highest = Most Anomalous) ---


,Avg 30d Volatility
2022,0.140610
2023,0.093637
2020,0.073427
2025,0.065251
2019,0.063986
2024,0.058314
2018,0.053433
2021,0.053018
2016,0.045721
2017,0.043867


In [5]:
from analysis.feature_creation import FeatureCreator

fc = FeatureCreator()
# 1. Get anomalies first
anomalies = fc.get_anomaly_years(result['combined'])

# 2. Generate the cleaned averages
df_enhanced = fc.get_cleaned_averages(result['combined'], anomalies)

# 3. Verify columns
print("Available Columns:", df_enhanced.columns.tolist())
display(df_enhanced[['Avg_5y_Clean', 'Avg_10y_Clean', 'Avg_15y_Clean']].tail())

Available Columns: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 'Avg_5y_Clean', 'Avg_10y_Clean', 'Avg_15y_Clean']


,Avg_5y_Clean,Avg_10y_Clean,Avg_15y_Clean
-4,0.7600,0.123333,0.066667
-3,0.6925,0.122917,0.099630
-2,-0.0400,0.124167,0.134815
-1,0.9575,0.149583,0.235556
0,1.2675,0.192500,0.443333


In [6]:
def export_to_excel(result, df_enhanced, filename="seasonality_analysis.xlsx"):
    # 1. Use the enhanced dataframe which already contains original years + cleaned averages
    df_export = df_enhanced.copy()

    # 2. Add the original STAT columns from the result dictionary
    df_export.index.name = "Days to Expiry"
    df_export['STAT_Average'] = result['average']
    df_export['STAT_Std_Dev'] = result.get('std')
    df_export['STAT_Rolling_2Sigma_Path'] = result.get('rolling_std_path')

    # 3. Sort index (usually -400 to 0)
    df_export = df_export.sort_index()

    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # Main data sheet
            df_export.to_excel(writer, sheet_name='Price Alignment')
            
            # Metadata/Warnings sheet
            if result.get('warnings'):
                pd.DataFrame(result['warnings'], columns=['Warnings']).to_excel(writer, sheet_name='Metadata')
        
        print(f"✅ Data successfully exported to {filename}")
    except Exception as e:
        print(f"❌ Failed to export Excel: {e}")

In [7]:
# 1. Calculate the rank column
current_rank_series = fc.calculate_current_rank(result['combined'])

# 2. Add it to your enhanced dataframe
df_enhanced['Current_Rank'] = current_rank_series

# 3. Export to Excel
export_to_excel(result, df_enhanced)

✅ Data successfully exported to seasonality_analysis.xlsx


In [8]:
export_to_excel(result, df_enhanced, filename="seasonality_analysis.xlsx")

✅ Data successfully exported to seasonality_analysis.xlsx


In [9]:
# 1. Initialize
fc = FeatureCreator()

# 2. Get Anomalies and Cleaned Averages (from previous steps)
anomalies = fc.get_anomaly_years(result['combined'])
df_enhanced = fc.get_cleaned_averages(result['combined'], anomalies)

# 3. Add Current Rank
df_enhanced['Current_Rank'] = fc.calculate_current_rank(result['combined'])

# 4. Add Win Rates and Expectancy
stats_df = fc.calculate_stats(result['combined'])
df_enhanced = df_enhanced.join(stats_df)

# 5. Export to Excel (using the updated function from before)
export_to_excel(result, df_enhanced)

✅ Data successfully exported to seasonality_analysis.xlsx
